In [38]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [39]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.rwf2000 import RWF2000Dataset
from src.baseline_cnn_lstm import BaselineCNNLSTM
from src.config import DATASET_ROOT
from src.config import CHECKPOINT_DIR
from tqdm.notebook import tqdm

In [40]:
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print(device)

cuda:2


In [41]:
hyperparameters = {
    "num_frames": 32, 
    "batch_size": 4,
    "hidden_size": 256,
    "learning_rate": 1e-4,
    "epochs": 15,
}

In [42]:
# dataset and loaders

train_dataset = RWF2000Dataset(DATASET_ROOT, split="train", num_frames=hyperparameters["num_frames"])
val_dataset = RWF2000Dataset(DATASET_ROOT, split="val", num_frames=hyperparameters["num_frames"])

train_loader = DataLoader(
    train_dataset,
    batch_size=hyperparameters["batch_size"],
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=hyperparameters["batch_size"],
    shuffle=False,
    num_workers=4
)

In [43]:
model = BaselineCNNLSTM(
    hidden_size=hyperparameters["hidden_size"],
    num_layers=1,
    num_classes=2,
    dropout=0.3,
    freeze_cnn=True
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=hyperparameters["learning_rate"]
)

In [44]:
# training function

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    
    for videos, labels in dataloader:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        outputs = model(videos)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * videos.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct/total:.4f}"
        )
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [45]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for videos, labels in dataloader:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(videos)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * videos.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{correct/total:.4f}"
            )
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [46]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

num_epochs = hyperparameters["epochs"]

for epoch in range(num_epochs):

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)    
    history["val_acc"].append(val_acc)
    
    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )


Epoch 1/15
--------------------------------------------------


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Validation:   0%|          | 0/100 [00:00<?, ?it/s]

Train Loss: 0.6413 | Train Acc: 0.6294 | Val Loss: 0.5675 | Val Acc: 0.6700

Epoch 2/15
--------------------------------------------------


Training:   0%|          | 0/400 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/100 [00:00<?, ?it/s]

Train Loss: 0.4090 | Train Acc: 0.8019 | Val Loss: 0.6065 | Val Acc: 0.7350

Epoch 15/15
--------------------------------------------------


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Validation:   0%|          | 0/100 [00:00<?, ?it/s]

Train Loss: 0.3988 | Train Acc: 0.8106 | Val Loss: 0.5466 | Val Acc: 0.7550


In [47]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "history": history,
    "config": hyperparameters
}

In [48]:
torch.save(
    checkpoint,
    CHECKPOINT_DIR / "baseline_cnn_lstm" / "baseline_cnn_lstm_v1.pt" 
)